# Reservoir Audio

Reservoir Audio cuts a sound into chunks, learns the order they come in with a quantum reservoir, and strings them into a new arrangement. Train once, then generate as many takes as you like from the trained model. (The API names it QRC Audio.)

## Setup

In [ ]:
import getpass
import os

from moth_atlas import Atlas

ENGINE = "qrc-audio-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "qrc-audio-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
import io
import wave

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display


def samples(wav):
    """A WAV file (path or bytes) as (float samples in [-1, 1], one column per channel; sample rate)."""
    with wave.open(wav if isinstance(wav, str) else io.BytesIO(wav)) as w:
        width, channels, rate = w.getsampwidth(), w.getnchannels(), w.getframerate()
        raw = np.frombuffer(w.readframes(w.getnframes()), np.uint8)
    if width == 3:  # 24-bit: pad each sample to 32 bits
        raw, width = np.pad(raw.reshape(-1, 3), ((0, 0), (1, 0))).reshape(-1), 4
    x = raw.view(f"<i{width}").astype(np.float32) / 2 ** (8 * width - 1)
    return x.reshape(-1, channels), rate


def listen(clips, play=True, stereo=False, rate=16000, height=1.1):
    """Plot and play {label: WAV path or bytes}, one row each. The players are resampled to `rate` (mono unless
    `stereo`) to keep this notebook small; the plots and downloads keep the full file."""
    fig, axes = plt.subplots(len(clips), 1, figsize=(9, height * len(clips)), sharex=True, squeeze=False)
    for ax, (label, wav) in zip(axes[:, 0], clips.items()):
        x, sr = samples(wav)
        t = np.arange(len(x)) / sr
        for c in range(x.shape[1]):
            ax.plot(t, x[:, c], lw=0.4, alpha=0.8)
        ax.set_ylim(-1, 1)
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center")
    axes[-1, 0].set_xlabel("seconds")
    plt.show()
    for label, wav in clips.items() if play else ():
        x, sr = samples(wav)
        x = x if stereo else x.mean(axis=1, keepdims=True)
        t = np.arange(0, len(x) / sr, 1 / rate)
        y = np.stack([np.interp(t, np.arange(len(x)) / sr, x[:, c]) for c in range(x.shape[1])])
        print(label)
        display(Audio(y if stereo else y[0], rate=rate, normalize=False))

def output_asset(job, slot):
    """One output file of a finished job, as an Asset you can pass straight into another job's files."""
    out = next(o for o in job.outputs if o.slot == slot)
    return client.assets.get(out.output_asset_id)


def arrangement(remix, source="loop.wav", chunk_seconds=0.5):
    """Which chunk of the source each chunk of a remix is (the closest match), as a list of chunk numbers."""
    x, rate = samples(source)
    y, _ = samples(remix)
    n = int(chunk_seconds * rate)
    chunks = [x[i:i + n, 0] for i in range(0, len(x) - n + 1, n)]
    return [int(np.argmin([np.sum((y[i:i + n, 0] - c) ** 2) for c in chunks])) for i in range(0, len(y) - n + 1, n)]

## Try it

`loop.wav` beside this notebook is 4 bars at 120 bpm made from Moth's QEcho sample pack: 16 beats, each with its own drum and key, so you can hear the reordering. At `chunk_seconds` 0.5 each chunk is one beat; the remix strings 16 of them back together. The numbers under the plot are which beat of the loop each beat of the remix is.

In [ ]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "chunk_seconds": 0.5,
        "length": 16,
        "quality": "fast",
        "variation": 1,
        "seed": 7,
    },
    files={
        "audio": "loop.wav",
    },
)
listen({"loop": "loop.wav", "remix": job.output_bytes("result")})
print("loop: ", list(range(16)))
print("remix:", arrangement(job.output_bytes("result")))

Keep the trained `model` and the chunk `vocabulary` from that run, and generate another take from them: no training this time.

In [ ]:
model, vocabulary = output_asset(job, "model"), output_asset(job, "vocabulary")

job = client.jobs.run(engine=ENGINE, params={"length": 16, "seed": 8}, files={"model": model, "chunks": vocabulary})
listen({"another take": job.output_bytes("result")})
print("remix:", arrangement(job.output_bytes("result")))

## Parameters

What to expect from each:

- `chunk_seconds`: the length of each chunk cut from `audio`. Match it to the beat (0.5 s at 120 bpm) for rhythmic results.
- `length` (up to 500): how many chunks the output strings together.
- `variation` (above 0): how freely the result departs from the learned order. Low is tight and repetitive, high loose and surprising.
- `quality`: how hard to train on a fresh piece: `instant` (no training, a quick shuffle), `fast`, `moderate` (default) or `complete` (slow and tight).
- `seed`: fixes a result. On a fresh piece it also fixes the learned reservoir; with a `model` it just picks a different take.
- `loop`: treat the piece as looping, so its end flows back into its start (learning only).
- `crossfade` (0 to 1000 ms): fades between chunks; 0 is hard cuts.
- `training_sequence`: advanced: chunk filenames in a specific order to learn instead of their natural order.

Inputs: `audio` to learn from, or `chunks` (a ZIP of WAVs, e.g. a previous `vocabulary`), plus `model` to skip training. Outputs: `result` (the WAV), `model`, `vocabulary` and `state`.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Tokens.** The input is cut into a vocabulary of tokens: here, `chunk_seconds`-long slices of the audio (the `vocabulary` output), and the piece becomes a sequence of them.

**Reservoir.** The sequence drives a quantum reservoir: a small, fixed quantum system whose state is nudged by each token in turn. Its measured features hold a fading memory of what came before, richer than the last token alone. The reservoir itself is never trained.

**Readout.** Only a simple readout is trained, mapping the reservoir's features to the next token. `quality` sets how hard it trains (`instant` skips training for a quick untrained shuffle). With `loop`, the end of the piece is treated as flowing back into its start.

**Generate.** Starting from the reservoir's state, the engine predicts the next token, samples one (`variation` sets how far it may stray from the most likely), feeds it back in and repeats `length` times.

The trained readout comes back as the `model` output: pass it into later runs to generate more takes without training again. The method follows *Level Generation with Quantum Reservoir Computing* (arXiv 2505.13287).

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Variation

Three takes from the same trained model: from close to the loop's own order to far from it.

In [ ]:
clips = {}
for variation in (0.2, 1.0, 3.0):
    print(f"variation={variation}")
    job = client.jobs.run(engine=ENGINE, params={"length": 8, "variation": variation, "seed": 1},
                          files={"model": model, "chunks": vocabulary}, quiet=True)
    clips[f"variation {variation}"] = job.output_bytes("result")
    print("  ", arrangement(clips[f"variation {variation}"]))
listen(clips)

### Quality

Fresh training each time: `instant` doesn't train at all, so its order is close to a shuffle; `complete` learns the loop tightly.

In [ ]:
clips = {}
for quality in ("instant", "fast", "complete"):
    print(f"quality={quality}")
    job = client.jobs.run(engine=ENGINE, params={"chunk_seconds": 0.5, "length": 8, "quality": quality, "seed": 7},
                          files={"audio": "loop.wav"}, quiet=True)
    clips[quality] = job.output_bytes("result")
    print("  ", arrangement(clips[quality]))
listen(clips, play=False)

### Crossfade

Hard cuts keep the beat crisp; longer crossfades blur the joins.

In [ ]:
clips = {}
for crossfade in (0, 40, 200):
    print(f"crossfade={crossfade}")
    job = client.jobs.run(engine=ENGINE, params={"length": 8, "crossfade": crossfade, "seed": 1},
                          files={"model": model, "chunks": vocabulary}, quiet=True)
    clips[f"crossfade {crossfade} ms"] = job.output_bytes("result")
listen(clips)

## Questions

**What makes this quantum?**

The memory: the reservoir is a quantum system, and its measured features, which carry the sequence's history, come from a quantum state much larger than its number of qubits suggests. Only the classical readout on top is trained.

**What kind of audio works best?**

Material with a clear pulse, cut on the beat: loops, grooves, phrases. Each chunk is reused whole, so chunk boundaries that fall mid-note will be audible.

**Can I reuse a model on different sounds?**

The model learns orderings of the vocabulary it was trained on. Pass the same `vocabulary` with it; a different ZIP of the same number of chunks would play new sounds in the learned order.

**Is this running on quantum hardware?**

No, the reservoir is simulated.

Sample sounds from Moth's QEcho sample pack (synthesised, no licence restrictions).